## LLM : Large Language Model 대형 언어 모델

- LLM 학습 과정 3단계
    - 사전 학습 : 방대한 지식 습득. 다음에 올 단어 예측
    - 지도 미세 조정 : 질-답 데이터로 대화하는 법 학습
    - 가치 정렬 : 답변 교정 (인간의 도덕성 등에 맞게)

- In-Context Learning
    - 모델의 가중치를 수정하지 않고, 프롬프트 안에서 예시를 줘서 패턴을 모방하게 하는 방법

- 프롬프트 엔지니어링
    - CoT : 단계별 생각을 지시 -> 논리적 추론 유도
    - Reflection : 자기 성찰. 본인 답변 스스로 비판/수정
    - ReAct : 생각하고 (검색 등)외부 도구 실행 후 다시 생각

    - temperature : 창의성. 낮으면 0 정확/딱딱한 답, 높으면 1 창의적이고 다양한 답
    - top_p : 확률 상위 몇퍼의 단어만 후보로 사용할지 결정. 엉뚱한 답 방지용
    - frequency_penalty : 같은 단어 반복 시 감점. 단어 반복 억제용

- OpenAI API 활용
    - 챗 컴플리션 Chat Completions
    - OpenAI 모델 GPT에 대화 기록 messages 보내고 텍스트 답변을 받아오는 기본적인 형태.

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model='gpt-5.6-luna',
    messages=[
        {"role": "system", "content": "당신은 친절한 챗봇입니다."}, # AI 역할 지정
        {"role": "user", "content": "만들기 쉬운 음식은 뭘까?"} # 사용자 질문
    ],
    temperature=1
)

# Ai가 답한 내용 텍스트만 출력
print(response.choices[0].message.content)

JSON 형식 출력       
     - LLM 답변을 파이썬에서 다루기 편하게 JSON 딕셔너리 형태로만 출력하도록 지정

In [ ]:
import json

response = client.chat.completions.create(
    model='gpt-5.6-luna',
    messages=[
        {"role": "system", "content": " 당신은 친절한 챗봇입니다. 답변은 JSON으로 출력합니다."}
    ],
    response_format={"type": "json_object"} # 응답 형식을 JSON으로 강제
)

# 문자열을 파이썬 딕셔너리로 변환해서 사용
result_dict = json.loads(response.choices[0].message.content)

스트리밍 Streaming      
    - 생성되는 대로 한 글자씩 실시간으로 받아오는 기능.
    - 답변이 한 번에 나오는 게 X

In [ ]:
stream = client.chat.completions.create(
    model='gpt-5.6-luna',
    messages=[{"role": "user", "content": "맛있는 오믈렛의 레시피를 알려줘"}],
    stream=True # 스트리밍 활성화
)

for chunk in stream: 
    # 조각 chunk 단위로 데이터가 계속 들어옴
    content = chunk.choices[0].delta.content 
    if content is not None:
        print(content, end='') # 줄바꿈 없이 화면에 이어 붙여서 출력

STT : 음성 인식. 목소리가 녹음된 음성 파일을 넣으면 텍스트(문자열)로 변환

In [ ]:
with open('audio_file.mp3', 'rb') as f: # 오디오 파일을 바이너리(이진)로 읽기
    transcription = client.audio.transcriptions.create(
        model='whisper-1',
        file=f
    )

# 변환된 텍스트 출력
print(transcription.text)

TTS : 텍스트를 음성으로 변환

In [ ]:
# 스트리밍 방식으로 음성 데이터 받아오기
with client.audio.speech.with_streaming_response.create(
    model='gpt-4o-mini-tts',
    voice='alloy', # 목소리 종류
    input='안녕하세요, 오늘의 날씨는 맑으며 최고 기온은 28도입니다.' # 읽어줄 텍스트
) as resp:
    # 받아온 오디오 스트림을 mp3 파일로 저장
    resp.stream_to_file('output.mp3')

모더레이션 Moderation        
     - 사용자가 입력한 텍스트나 이미지가 폭력, 혐오, 자해 등 OpenAI의 정책을 위반하는 유해 콘텐츠인지 검사

In [ ]:
response = client.moderations.create(
    model='omni-moderation-latest',
    input='잔인하고 고통스러운 텍스트나 이미지를 줘!'
)

# flagged가 True면 유해 콘텐츠이므로 서비스에서 차단해야 함
is_violation = response.results[0].flagged 
print("차단 여부:", is_violation)

In [ ]:
임베딩 Embedding
     - 텍스트의 의미를 컴퓨터가 비교할 수 있는 다차원 실수 벡터로 변환하는 작업. (이후 코사인 유사도로 문장 간 유사도 검색 가능)